In [1]:
import json
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# SETUP DIRECTORY PATHS & DUCKDB CONNECTION
# ==========================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
RESULT_DIR = BASE_DIR / "results"
FIG_DIR = RESULT_DIR / "figures"
ANALYSIS_DIR = BASE_DIR / "analysis"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = RESULT_DIR / "step03_01_star_schema.duckdb"
assert DB_PATH.exists(), f"Warehouse database not found at {DB_PATH}"

con = duckdb.connect(str(DB_PATH))

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

FINDINGS = {}

counts_check = q("""
SELECT 
    (SELECT COUNT(*) FROM vw_olap_observed_arrival) AS observed_arrival_rows,
    (SELECT COUNT(*) FROM vw_olap_observed_origin) AS observed_origin_rows,
    (SELECT COUNT(*) FROM vw_olap_observed) AS observed_total_rows,
    (SELECT COUNT(*) FROM vw_olap_imputed) AS imputed_total_rows;
""")
print("=== STEP 3.4: PUNCTUALITY & TEMPORAL PATTERN DISCOVERY ===")
print(counts_check.to_string(index=False))
FINDINGS["baseline_denominators"] = counts_check.iloc[0].to_dict()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== STEP 3.4: PUNCTUALITY & TEMPORAL PATTERN DISCOVERY ===
 observed_arrival_rows  observed_origin_rows  observed_total_rows  imputed_total_rows
              33773361               1808957             35582318             1032506


In [2]:
# ==========================================
# PATTERN 1: ARRIVAL DELAY (station_no > 1) VS ORIGIN DEPARTURE DELAY (station_no = 1)
# ==========================================
print("\n=== PATTERN 1A: ARRIVAL DELAY METRICS BY TRAIN CATEGORY (station_no > 1, is_imputed = FALSE) ===")

arrival_by_type_df = q("""
SELECT 
    primary_type_code,
    COUNT(*) AS observed_arrival_stops,
    COUNT(DISTINCT train_no) AS distinct_trains,
    ROUND(AVG(delay), 2) AS mean_arrival_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_arrival_delay_min,
    ROUND(STDDEV(delay), 2) AS std_arrival_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_arrival_delay_min,
    ROUND(STDDEV(delay) / NULLIF(AVG(delay), 0), 4) AS cv_arrival_delay,
    ROUND(QUANTILE_CONT(delay, 0.90), 2) AS p90_arrival_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.95), 2) AS p95_arrival_delay_min,
    ROUND(AVG(is_early) * 100.0, 2) AS early_arrival_lt_0m_pct,
    ROUND(AVG(is_exact_ontime) * 100.0, 2) AS exact_ontime_0m_pct,
    ROUND(AVG(is_early_or_ontime) * 100.0, 2) AS early_or_ontime_le_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_moderate_delay_15_60m) * 100.0, 2) AS moderate_delay_15_60m_pct,
    ROUND(AVG(is_severe_delay_gt_60m) * 100.0, 2) AS severe_delay_gt_60m_pct,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed_arrival
GROUP BY primary_type_code
ORDER BY observed_arrival_stops DESC;
""")
print(arrival_by_type_df.to_string(index=False))
arrival_by_type_df.to_csv(RESULT_DIR / "olap_step03_04_arrival_delay_by_train_category.csv", index=False)

print("\n=== PATTERN 1B: ORIGIN DEPARTURE DELAY METRICS BY TRAIN CATEGORY (station_no = 1, is_imputed = FALSE) ===")
origin_by_type_df = q("""
SELECT 
    primary_type_code,
    COUNT(*) AS observed_origin_departures,
    COUNT(DISTINCT train_no) AS distinct_trains,
    ROUND(AVG(delay), 2) AS mean_departure_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_departure_delay_min,
    ROUND(STDDEV(delay), 2) AS std_departure_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_departure_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.90), 2) AS p90_departure_delay_min,
    ROUND(AVG(is_early) * 100.0, 4) AS early_departure_lt_0m_pct,
    ROUND(AVG(is_exact_ontime) * 100.0, 2) AS exact_ontime_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM vw_olap_observed_origin
GROUP BY primary_type_code
ORDER BY observed_origin_departures DESC;
""")
print(origin_by_type_df.to_string(index=False))
origin_by_type_df.to_csv(RESULT_DIR / "olap_step03_04_origin_departure_by_train_category.csv", index=False)

print("\n=== PATTERN 1C: ARRIVAL DELAY METRICS BY RAILWAY ZONE (station_no > 1, is_imputed = FALSE) ===")
arrival_by_zone_df = q("""
SELECT 
    station_zone,
    COUNT(*) AS observed_arrival_stops,
    ROUND(AVG(delay), 2) AS mean_arrival_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_arrival_delay_min,
    ROUND(STDDEV(delay), 2) AS std_arrival_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_arrival_delay_min,
    ROUND(AVG(is_early) * 100.0, 2) AS early_arrival_lt_0m_pct,
    ROUND(AVG(is_exact_ontime) * 100.0, 2) AS exact_ontime_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed_arrival
GROUP BY station_zone
ORDER BY bottleneck_gt_2sd_pct DESC;
""")
print(arrival_by_zone_df.to_string(index=False))
arrival_by_zone_df.to_csv(RESULT_DIR / "olap_step03_04_arrival_delay_by_zone.csv", index=False)

FINDINGS["pattern1_arrival_vs_origin"] = {
    "arrival_by_train_category": arrival_by_type_df.to_dict(orient="records"),
    "origin_by_train_category": origin_by_type_df.to_dict(orient="records"),
    "arrival_by_zone": arrival_by_zone_df.to_dict(orient="records")
}


=== PATTERN 1A: ARRIVAL DELAY METRICS BY TRAIN CATEGORY (station_no > 1, is_imputed = FALSE) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

primary_type_code  observed_arrival_stops  distinct_trains  mean_arrival_delay_min  median_arrival_delay_min  std_arrival_delay_min  iqr_arrival_delay_min  cv_arrival_delay  p90_arrival_delay_min  p95_arrival_delay_min  early_arrival_lt_0m_pct  exact_ontime_0m_pct  early_or_ontime_le_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  moderate_delay_15_60m_pct  severe_delay_gt_60m_pct  high_risk_2sd_3sd_pct  critical_gt_3sd_pct  bottleneck_gt_2sd_pct
      PASS-TRAINS                16487821             3126                   33.65                      16.0                  55.88                   38.0            1.6605                   88.0                  127.0                     2.21                14.50                      16.71               49.67               50.33                      33.36                    16.97                 1.5723               1.0400                 2.6123
       EXP-TRAINS                12150217             2071                   38.26                

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

station_zone  observed_arrival_stops  mean_arrival_delay_min  median_arrival_delay_min  std_arrival_delay_min  iqr_arrival_delay_min  early_arrival_lt_0m_pct  exact_ontime_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  high_risk_2sd_3sd_pct  critical_gt_3sd_pct  bottleneck_gt_2sd_pct
        SCOR                      46                   97.78                      44.5                 144.31                  88.75                     0.00                15.22               39.13               60.87                 4.3478              13.0435                17.3913
        SECR                 1226260                   59.61                      31.0                  88.38                  67.00                     0.96                11.36               34.19               65.81                 4.0058               4.0129                 8.0187
         SER                 1420049                   55.78                      19.0                 110.45                  60.00          

In [3]:
# ==========================================
# PATTERN 2: ROUTE PROGRESSION, DISTANCE BANDS & SCHEDULE PADDING (EARLY ARRIVALS)
# ==========================================
print("\n=== PATTERN 2A: DELAY ACCUMULATION ALONG STOP POSITION BUCKETS (vw_olap_observed) ===")

stop_progression_df = q("""
WITH StopTagged AS (
    SELECT 
        *,
        MAX(station_no) OVER (PARTITION BY date_id, train_no) AS journey_max_stop
    FROM vw_olap_observed
),
Bucketed AS (
    SELECT 
        *,
        CASE 
            WHEN station_no = 1 THEN '1_Origin (Stop 1)'
            WHEN station_no = journey_max_stop THEN '7_Terminal Destination (Final Stop)'
            WHEN station_no BETWEEN 2 AND 5 THEN '2_Early Route (Stops 2-5)'
            WHEN station_no BETWEEN 6 AND 10 THEN '3_Mid-Early Route (Stops 6-10)'
            WHEN station_no BETWEEN 11 AND 20 THEN '4_Mid Route (Stops 11-20)'
            WHEN station_no BETWEEN 21 AND 35 THEN '5_Mid-Late Route (Stops 21-35)'
            ELSE '6_Long-Haul Tail (Stops 36+)'
        END AS route_stage
    FROM StopTagged
)
SELECT 
    route_stage,
    COUNT(*) AS observed_stops,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS share_pct,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_delay_min,
    ROUND(AVG(is_early) * 100.0, 2) AS early_lt_0m_pct,
    ROUND(AVG(is_exact_ontime) * 100.0, 2) AS exact_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM Bucketed
GROUP BY route_stage
ORDER BY route_stage;
""")
print(stop_progression_df.to_string(index=False))
stop_progression_df.to_csv(RESULT_DIR / "olap_step03_04_route_stage_progression.csv", index=False)

print("\n=== PATTERN 2B: ARRIVAL DELAY BY DISTANCE-FROM-ORIGIN BANDS (station_no > 1, has_schedule = TRUE) ===")
distance_progression_df = q("""
WITH DistBand AS (
    SELECT 
        *,
        CASE 
            WHEN distance_from_origin <= 100 THEN '1_001-100 km'
            WHEN distance_from_origin <= 250 THEN '2_101-250 km'
            WHEN distance_from_origin <= 500 THEN '3_251-500 km'
            WHEN distance_from_origin <= 1000 THEN '4_501-1000 km'
            WHEN distance_from_origin <= 1500 THEN '5_1001-1500 km'
            ELSE '6_1500+ km'
        END AS distance_band
    FROM vw_olap_observed_arrival
    WHERE has_schedule = TRUE
)
SELECT 
    distance_band,
    COUNT(*) AS observed_arrival_stops,
    ROUND(AVG(delay), 2) AS mean_arrival_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_arrival_delay_min,
    ROUND(STDDEV(delay), 2) AS std_arrival_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_arrival_delay_min,
    ROUND(AVG(is_early) * 100.0, 2) AS early_lt_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM DistBand
GROUP BY distance_band
ORDER BY distance_band;
""")
print(distance_progression_df.to_string(index=False))
distance_progression_df.to_csv(RESULT_DIR / "olap_step03_04_distance_band_progression.csv", index=False)

FINDINGS["pattern2_route_and_distance_progression"] = {
    "by_route_stage": stop_progression_df.to_dict(orient="records"),
    "by_distance_band": distance_progression_df.to_dict(orient="records")
}


=== PATTERN 2A: DELAY ACCUMULATION ALONG STOP POSITION BUCKETS (vw_olap_observed) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                        route_stage  observed_stops  share_pct  mean_delay_min  median_delay_min  std_delay_min  iqr_delay_min  early_lt_0m_pct  exact_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  bottleneck_gt_2sd_pct  critical_gt_3sd_pct
                  1_Origin (Stop 1)         1808957       5.08           10.73               1.0          38.78            4.0             0.01         42.61               86.56               13.44                 1.0436               0.5455
          2_Early Route (Stops 2-5)         6922471      19.45           20.69               9.0          46.41           19.0             0.00         17.54               66.69               33.31                 1.4616               0.7740
     3_Mid-Early Route (Stops 6-10)         7518373      21.13           30.06              16.0          54.90           29.0             0.00         11.76               49.60               50.40                 2.1398               1.1265
          4_Mid Route (Stops 11-

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 distance_band  observed_arrival_stops  mean_arrival_delay_min  median_arrival_delay_min  std_arrival_delay_min  iqr_arrival_delay_min  early_lt_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  bottleneck_gt_2sd_pct  critical_gt_3sd_pct
  1_001-100 km                14676473                   25.64                      12.0                  47.30                   27.0             1.44               57.43               42.57                 1.5991               0.7055
  2_101-250 km                 8624154                   38.44                      22.0                  57.96                   41.0             2.01               40.32               59.68                 2.9910               1.2507
  3_251-500 km                 4559098                   38.64                      21.0                  61.59                   40.0             2.10               41.79               58.21                 3.4377               1.6413
 4_501-1000 km                 3269013                  

In [4]:
# ==========================================
# PATTERN 3: DIURNAL (SCHEDULED HOUR-OF-DAY) ARRIVAL DELAY PATTERN
# ==========================================
print("\n=== PATTERN 3: ARRIVAL DELAY BY SCHEDULED ARRIVAL TIME-OF-DAY BLOCK ===")

diurnal_df = q("""
WITH HourExtracted AS (
    SELECT 
        *,
        CAST(SPLIT_PART(CAST(arrival_time AS VARCHAR), ':', 1) AS INTEGER) AS sched_arr_hour
    FROM vw_olap_observed_arrival
    WHERE has_schedule = TRUE
),
TimeBlocks AS (
    SELECT 
        *,
        CASE 
            WHEN sched_arr_hour BETWEEN 0 AND 3 THEN '1_Late Night (00:00-03:59)'
            WHEN sched_arr_hour BETWEEN 4 AND 7 THEN '2_Early Morning (04:00-07:59)'
            WHEN sched_arr_hour BETWEEN 8 AND 11 THEN '3_Morning Peak (08:00-11:59)'
            WHEN sched_arr_hour BETWEEN 12 AND 15 THEN '4_Afternoon (12:00-15:59)'
            WHEN sched_arr_hour BETWEEN 16 AND 19 THEN '5_Evening Peak (16:00-19:59)'
            ELSE '6_Night (20:00-23:59)'
        END AS time_of_day_block
    FROM HourExtracted
)
SELECT 
    time_of_day_block,
    COUNT(*) AS observed_arrival_stops,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS traffic_share_pct,
    ROUND(AVG(delay), 2) AS mean_arrival_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_arrival_delay_min,
    ROUND(STDDEV(delay), 2) AS std_arrival_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_arrival_delay_min,
    ROUND(AVG(is_early) * 100.0, 2) AS early_lt_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM TimeBlocks
GROUP BY time_of_day_block
ORDER BY time_of_day_block;
""")
print(diurnal_df.to_string(index=False))
diurnal_df.to_csv(RESULT_DIR / "olap_step03_04_diurnal_time_of_day.csv", index=False)

FINDINGS["pattern3_diurnal_time_of_day"] = diurnal_df.to_dict(orient="records")


=== PATTERN 3: ARRIVAL DELAY BY SCHEDULED ARRIVAL TIME-OF-DAY BLOCK ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            time_of_day_block  observed_arrival_stops  traffic_share_pct  mean_arrival_delay_min  median_arrival_delay_min  std_arrival_delay_min  iqr_arrival_delay_min  early_lt_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  bottleneck_gt_2sd_pct  critical_gt_3sd_pct
   1_Late Night (00:00-03:59)                 2653303               7.92                   43.56                      21.0                  76.17                   42.0             0.74               42.07               57.93                 4.9709               2.7117
2_Early Morning (04:00-07:59)                 6185661              18.47                   30.38                      13.0                  60.99                   28.0             1.26               54.39               45.61                 2.7797               1.5464
 3_Morning Peak (08:00-11:59)                 7056055              21.07                   34.67                      17.0                  63.14                   36.0             2.13     

In [5]:
# ==========================================
# PATTERN 4: STATION DELAY AMPLIFICATION VS RECOVERY (station_no > 1, >= 1,000 STOPS)
# ==========================================
print("\n=== PATTERN 4: STATION-TO-STATION DELAY AMPLIFICATION (DELTA DELAY) ===")

con.execute("""
CREATE OR REPLACE TEMP TABLE station_delta_cube AS
WITH ConsecutiveStops AS (
    SELECT 
        date_id,
        train_no,
        station_no,
        station_name,
        station_full_name,
        station_zone,
        delay,
        is_imputed,
        is_bottleneck_gt_2sd,
        is_critical_only,
        LAG(station_no) OVER (PARTITION BY date_id, train_no ORDER BY station_no) AS prev_station_no,
        LAG(delay) OVER (PARTITION BY date_id, train_no ORDER BY station_no) AS prev_delay,
        LAG(is_imputed) OVER (PARTITION BY date_id, train_no ORDER BY station_no) AS prev_is_imputed
    FROM vw_olap_base
)
SELECT 
    station_name,
    station_full_name,
    station_zone,
    COUNT(*) AS observed_transitions,
    COUNT(DISTINCT train_no) AS distinct_trains,
    ROUND(AVG(delay), 2) AS mean_arrival_delay_min,
    ROUND(AVG(prev_delay), 2) AS mean_upstream_delay_min,
    ROUND(AVG(delay - prev_delay), 2) AS mean_delay_delta_min,
    ROUND(MEDIAN(delay - prev_delay), 2) AS median_delay_delta_min,
    ROUND(STDDEV(delay - prev_delay), 2) AS std_delay_delta_min,
    ROUND(AVG(CASE WHEN (delay - prev_delay) > 5 THEN 1.0 ELSE 0.0 END) * 100.0, 2) AS amplify_gt_5m_pct,
    ROUND(AVG(CASE WHEN (delay - prev_delay) > 15 THEN 1.0 ELSE 0.0 END) * 100.0, 2) AS surge_gt_15m_pct,
    ROUND(AVG(CASE WHEN (delay - prev_delay) < -5 THEN 1.0 ELSE 0.0 END) * 100.0, 2) AS recovery_lt_minus5m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS arrival_bottleneck_gt_2sd_pct
FROM ConsecutiveStops
WHERE is_imputed = FALSE 
  AND prev_is_imputed = FALSE
  AND station_no = prev_station_no + 1
GROUP BY station_name, station_full_name, station_zone
HAVING COUNT(*) >= 1000;
""")

top20_amplifiers = q("""
SELECT * FROM station_delta_cube
ORDER BY mean_delay_delta_min DESC
LIMIT 20;
""")
print("Top 20 Delay-Amplifying Stations (Highest Mean +Delta Delay, >= 1,000 Observed Transitions):")
print(top20_amplifiers.to_string(index=False))

top20_recoverers = q("""
SELECT * FROM station_delta_cube
ORDER BY mean_delay_delta_min ASC
LIMIT 20;
""")
print("\nTop 20 Delay-Absorbing / Buffer Recovery Stations (Most Negative Mean Delta Delay, >= 1,000 Transitions):")
print(top20_recoverers.to_string(index=False))

top20_amplifiers.to_csv(RESULT_DIR / "olap_step03_04_station_top20_delay_amplifiers_ge1000.csv", index=False)
top20_recoverers.to_csv(RESULT_DIR / "olap_step03_04_station_top20_delay_recoverers_ge1000.csv", index=False)

FINDINGS["pattern4_station_delay_amplification"] = {
    "top20_amplifiers_ge1000": top20_amplifiers.to_dict(orient="records"),
    "top20_recoverers_ge1000": top20_recoverers.to_dict(orient="records")
}


=== PATTERN 4: STATION-TO-STATION DELAY AMPLIFICATION (DELTA DELAY) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Top 20 Delay-Amplifying Stations (Highest Mean +Delta Delay, >= 1,000 Observed Transitions):
station_name station_full_name station_zone  observed_transitions  distinct_trains  mean_arrival_delay_min  mean_upstream_delay_min  mean_delay_delta_min  median_delay_delta_min  std_delay_delta_min  amplify_gt_5m_pct  surge_gt_15m_pct  recovery_lt_minus5m_pct  arrival_bottleneck_gt_2sd_pct
        SBGB     SUTEI BADAGAN          SER                  2161                6                   52.50                     3.53                 48.97                     0.0               184.86              11.24             10.32                     7.03                         8.0056
         MQN       MANYAMKONDA          SCR                  2837                9                   87.14                    45.08                 42.07                    34.0                40.37              77.09             67.96                     4.83                        10.2221
         BTQ           BETNOTI 

In [6]:
# ==========================================
# PATTERN 5: DIFFERENCES BETWEEN OBSERVED AND IMPUTED RECORDS
# ==========================================
print("\n=== PATTERN 5A: OBSERVED VS IMPUTED BY STOP CONTEXT (ORIGIN VS ARRIVAL) ===")

obs_vs_imp_context_df = q("""
SELECT 
    CASE WHEN station_no = 1 THEN 'Origin (station_no = 1)' ELSE 'Arrival (station_no > 1)' END AS stop_context,
    is_imputed,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY (station_no = 1)), 4) AS share_within_context_pct,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_delay_min,
    ROUND(AVG(is_early) * 100.0, 2) AS early_lt_0m_pct,
    ROUND(AVG(is_exact_ontime) * 100.0, 2) AS exact_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM vw_olap_base
GROUP BY (station_no = 1), is_imputed
ORDER BY stop_context DESC, is_imputed ASC;
""")
print(obs_vs_imp_context_df.to_string(index=False))
obs_vs_imp_context_df.to_csv(RESULT_DIR / "olap_step03_04_obs_vs_imp_by_stop_context.csv", index=False)

print("\n=== PATTERN 5B: OBSERVED VS IMPUTED BY TRAIN CATEGORY ===")
obs_vs_imp_type_df = q("""
SELECT 
    primary_type_code,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS obs_rows,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS imp_rows,
    ROUND(COUNT(*) FILTER (WHERE is_imputed = TRUE) * 100.0 / COUNT(*), 2) AS imp_share_pct,
    ROUND(AVG(delay) FILTER (WHERE is_imputed = FALSE), 2) AS obs_mean_delay,
    ROUND(AVG(delay) FILTER (WHERE is_imputed = TRUE), 2) AS imp_mean_delay,
    ROUND(MEDIAN(delay) FILTER (WHERE is_imputed = FALSE), 2) AS obs_median_delay,
    ROUND(MEDIAN(delay) FILTER (WHERE is_imputed = TRUE), 2) AS imp_median_delay,
    ROUND(STDDEV(delay) FILTER (WHERE is_imputed = FALSE), 2) AS obs_std_delay,
    ROUND(STDDEV(delay) FILTER (WHERE is_imputed = TRUE), 2) AS imp_std_delay,
    ROUND(AVG(is_on_time_15m) FILTER (WHERE is_imputed = FALSE) * 100.0, 2) AS obs_ontime_15m_pct,
    ROUND(AVG(is_on_time_15m) FILTER (WHERE is_imputed = TRUE) * 100.0, 2) AS imp_ontime_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) FILTER (WHERE is_imputed = FALSE) * 100.0, 4) AS obs_bottleneck_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) FILTER (WHERE is_imputed = TRUE) * 100.0, 4) AS imp_bottleneck_pct
FROM vw_olap_base
GROUP BY primary_type_code
ORDER BY obs_rows DESC;
""")
print(obs_vs_imp_type_df.to_string(index=False))
obs_vs_imp_type_df.to_csv(RESULT_DIR / "olap_step03_04_obs_vs_imp_by_train_category.csv", index=False)

print("\n=== PATTERN 5C: OBSERVED VS IMPUTED BY MONTH ===")
obs_vs_imp_month_df = q("""
SELECT 
    year,
    month,
    month_name,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS obs_rows,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS imp_rows,
    ROUND(COUNT(*) FILTER (WHERE is_imputed = TRUE) * 100.0 / COUNT(*), 2) AS imp_share_pct,
    ROUND(AVG(delay) FILTER (WHERE is_imputed = FALSE), 2) AS obs_mean_delay,
    ROUND(AVG(delay) FILTER (WHERE is_imputed = TRUE), 2) AS imp_mean_delay,
    ROUND(MEDIAN(delay) FILTER (WHERE is_imputed = FALSE), 2) AS obs_median_delay,
    ROUND(MEDIAN(delay) FILTER (WHERE is_imputed = TRUE), 2) AS imp_median_delay,
    ROUND(AVG(is_bottleneck_gt_2sd) FILTER (WHERE is_imputed = FALSE) * 100.0, 4) AS obs_bottleneck_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) FILTER (WHERE is_imputed = TRUE) * 100.0, 4) AS imp_bottleneck_pct
FROM vw_olap_base
GROUP BY year, month, month_name
ORDER BY year, month;
""")
print(obs_vs_imp_month_df.to_string(index=False))
obs_vs_imp_month_df.to_csv(RESULT_DIR / "olap_step03_04_obs_vs_imp_by_month.csv", index=False)

FINDINGS["pattern5_observed_vs_imputed"] = {
    "by_stop_context": obs_vs_imp_context_df.to_dict(orient="records"),
    "by_train_category": obs_vs_imp_type_df.to_dict(orient="records"),
    "by_month": obs_vs_imp_month_df.to_dict(orient="records")
}


=== PATTERN 5A: OBSERVED VS IMPUTED BY STOP CONTEXT (ORIGIN VS ARRIVAL) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            stop_context  is_imputed   n_rows  share_within_context_pct  mean_delay_min  median_delay_min  std_delay_min  iqr_delay_min  early_lt_0m_pct  exact_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  bottleneck_gt_2sd_pct  critical_gt_3sd_pct
 Origin (station_no = 1)       False  1808957                   97.7102           10.73               1.0          38.78            4.0             0.01         42.61               86.56               13.44                 1.0436               0.5455
 Origin (station_no = 1)        True    42392                    2.2898            6.45               0.0          24.63            2.0             0.00         50.33               92.25                7.75                 0.1250               0.1227
Arrival (station_no > 1)       False 33773361                   97.1519           37.01              17.0          66.73           37.0             1.73         11.88               47.36               52.64                 3.6011               1.8

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

primary_type_code  obs_rows  imp_rows  imp_share_pct  obs_mean_delay  imp_mean_delay  obs_median_delay  imp_median_delay  obs_std_delay  imp_std_delay  obs_ontime_15m_pct  imp_ontime_15m_pct  obs_bottleneck_pct  imp_bottleneck_pct
      PASS-TRAINS  17519157    224634           1.27           32.41           22.94              15.0               5.0          55.13          50.31               51.58               66.08              2.5148              1.5207
       EXP-TRAINS  12605410    572218           4.34           37.14           34.55              18.0              14.0          66.90          71.29               46.29               52.60              3.7395              3.8961
        SF-TRAINS   4690135    198303           4.06           43.68           39.92              17.0              14.0          87.53          86.34               46.98               53.15              6.0042              5.4886
       T18-TRAINS    321670     16996           5.02           14.71        

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 year  month month_name  obs_rows  imp_rows  imp_share_pct  obs_mean_delay  imp_mean_delay  obs_median_delay  imp_median_delay  obs_bottleneck_pct  imp_bottleneck_pct
 2025      2   February   1892863     79463           4.03           41.23           40.74              16.0              11.0              4.8418              5.7498
 2025      3      March   2936944    114678           3.76           34.27           30.29              15.0              11.0              3.2352              3.4087
 2025      4      April   2928999    109156           3.59           31.43           27.10              14.0               9.0              2.6735              2.5166
 2025      5        May   3037574    108882           3.46           35.66           31.71              15.0              10.0              3.5592              3.3302
 2025      6       June   2943199     99750           3.28           34.14           30.08              16.0              11.0              3.0913              3.079

In [7]:
# ==========================================
# STEP 6: GENERATE VISUALIZATIONS & EXPORT FINDINGS
# ==========================================
print("\n=== STEP 6: GENERATING STEP 3.4 VISUALIZATIONS IN results/figures/ ===")

plt.rcParams["figure.dpi"] = 150

# 1. Route Stage Progression: Delay Accumulation & Early Arrival Padding at Terminals
fig, ax1 = plt.subplots(figsize=(10, 5))
stages = [s.split("_", 1)[1] for s in stop_progression_df["route_stage"]]
x_s = range(len(stages))
ax1.plot(list(x_s), stop_progression_df["mean_delay_min"], marker="o", linewidth=2, color="#1f78b4", label="Mean Delay (min)")
ax1.plot(list(x_s), stop_progression_df["median_delay_min"], marker="s", linewidth=2, linestyle="--", color="#33a02c", label="Median Delay (min)")
ax1.set_xticks(list(x_s))
ax1.set_xticklabels(stages, rotation=25, ha="right")
ax1.set_ylabel("Delay (Minutes)")
ax1.legend(loc="upper left")
ax2 = ax1.twinx()
ax2.bar(list(x_s), stop_progression_df["early_lt_0m_pct"], alpha=0.3, color="#6a3d9a", width=0.45, label="Early Arrival <0m (%)")
ax2.set_ylabel("Early Arrival Rate (%)", color="#6a3d9a")
ax2.legend(loc="upper right")
plt.title("Route Progression: Delay Accumulation & Terminal Schedule Padding (Early Arrivals)")
fig.tight_layout()
f1 = FIG_DIR / "step03_04_route_stage_progression.png"
plt.savefig(f1)
plt.close(fig)

# 2. Diurnal Time-of-Day Pattern
fig, ax1 = plt.subplots(figsize=(9, 5))
t_blocks = [b.split("_", 1)[1] for b in diurnal_df["time_of_day_block"]]
x_t = range(len(t_blocks))
ax1.bar(list(x_t), diurnal_df["mean_arrival_delay_min"], width=0.45, color="#2b5c8f", label="Mean Arrival Delay (min)")
ax1.set_xticks(list(x_t))
ax1.set_xticklabels(t_blocks, rotation=20, ha="right")
ax1.set_ylabel("Mean Arrival Delay (Minutes)", color="#2b5c8f")
ax2 = ax1.twinx()
ax2.plot(list(x_t), diurnal_df["delayed_gt_15m_pct"], marker="o", linewidth=2, color="#d95f02", label="Delayed >15m (%)")
ax2.set_ylabel("Delayed >15m (%)", color="#d95f02")
plt.title("Diurnal Pattern: Scheduled Arrival Time-of-Day vs. Arrival Delay (station_no > 1)")
fig.tight_layout()
f2 = FIG_DIR / "step03_04_diurnal_arrival_delay.png"
plt.savefig(f2)
plt.close(fig)

# 3. Observed vs Imputed Mean Delay & Imputation Share by Train Category
fig, ax1 = plt.subplots(figsize=(10, 5))
x_c = range(len(obs_vs_imp_type_df))
w = 0.35
ax1.bar([i - w/2 for i in x_c], obs_vs_imp_type_df["obs_mean_delay"], width=w, label="Observed Mean Delay (min)", color="#1f78b4")
ax1.bar([i + w/2 for i in x_c], obs_vs_imp_type_df["imp_mean_delay"], width=w, label="Imputed Mean Delay (min)", color="#ff7f00")
ax1.set_xticks(list(x_c))
ax1.set_xticklabels(obs_vs_imp_type_df["primary_type_code"], rotation=25, ha="right")
ax1.set_ylabel("Mean Delay (Minutes)")
ax1.legend(loc="upper left")
ax2 = ax1.twinx()
ax2.plot(list(x_c), obs_vs_imp_type_df["imp_share_pct"], color="#6a3d9a", marker="D", linestyle=":", linewidth=2, label="Imputed Share (%)")
ax2.set_ylabel("Imputed Share of Category (%)", color="#6a3d9a")
ax2.legend(loc="upper right")
plt.title("Observed vs. Imputed Records by Train Category")
fig.tight_layout()
f3 = FIG_DIR / "step03_04_observed_vs_imputed_train_category.png"
plt.savefig(f3)
plt.close(fig)

print(f"-> Saved figures:\n   {f1.name}\n   {f2.name}\n   {f3.name}")

findings_path = RESULT_DIR / "step03_04_OLAP_FINDINGS.json"
with open(findings_path, "w") as f:
    json.dump(FINDINGS, f, indent=4)

print(f"\n-> Saved Step 3.4 pattern discovery findings to {findings_path.as_posix()}")
con.close()


=== STEP 6: GENERATING STEP 3.4 VISUALIZATIONS IN results/figures/ ===
-> Saved figures:
   step03_04_route_stage_progression.png
   step03_04_diurnal_arrival_delay.png
   step03_04_observed_vs_imputed_train_category.png

-> Saved Step 3.4 pattern discovery findings to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step03_04_OLAP_FINDINGS.json
